# Modern Transport Profiles

This notebook shows the current transport postprocessing workflow built on the modern API.

It covers:
- flux-surface reduced Bohm, gyro-Bohm, and mixed transport profiles
- collisionality-dependent pinch profile built from the same reduced quantities
- projection of 1D surface-derived profiles back onto the 2D solution for plotting

At this stage the transport model is the surface-based reference implementation. A later comparison notebook can add the fully local 2D diagnostic version on top of the same plots.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from hdg_postprocess.api import (
    configure_solution_setup,
    load_reference_element,
    load_solution,
)


## Load the demo solution

This uses the same `power_balance_with_cooling` case as the focused transport tests.


In [ ]:
repo_root = Path.cwd().resolve()
while not (repo_root / "hdg_postprocess").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

demo_data = repo_root / "demos" / "data"
case_name = "limiter_case"

solution = load_solution(
    str(demo_data / "solutions" / "limiter_case") + "/",
    "Sol2D_WEST_60527_P8_DPe0.100E+01_DPai0.314E+06_DPae0.105E+08_0000",
    n_partitions=1,
)

solution.mesh.metadata.reference_element = load_reference_element(
    str(demo_data / "reference_elements" / "reference_triangle_P8.mat")
)

configure_solution_setup(
    solution,
    reference_element=str(demo_data / "reference_elements" / "reference_triangle_P8.mat"),
    neutral_diffusion=True,
)


## Choose a flux-surface grid and transport settings

The default derivative mode now uses HDG gradients projected onto the flux-normal direction.


In [ ]:
rho_full_max = float(np.nanmax(solution.flux_surface.rho(target="node")))
rho = np.linspace(0.0, rho_full_max, 96)
method = "gauss_shell"
width = 2e-3
rho_inner = 0.8
rho_edge = 0.99
derivative_mode = "flux_normal"


## Compute surface-based transport and pinch profiles

`delta_te` is still the same non-local edge quantity used by the canonical mixed Bohm/gyro-Bohm model.


In [ ]:
bohm = solution.transport.bohm(
    rho,
    rho_inner=rho_inner,
    rho_edge=rho_edge,
    method=method,
    width=width,
    derivative_mode=derivative_mode,
)

gyrobohm = solution.transport.gyrobohm(
    rho,
    rho_inner=rho_inner,
    rho_edge=rho_edge,
    method=method,
    width=width,
    derivative_mode=derivative_mode,
)

mixed = solution.transport.bohm_gyrobohm(
    rho,
    rho_inner=rho_inner,
    rho_edge=rho_edge,
    method=method,
    width=width,
    derivative_mode=derivative_mode,
)

pinch_militello = solution.flux_surface.pinch_velocity(
    rho,
    mixed["diffusion"],
    model="militello",
    coefficient=0.5,
    rho_edge=rho_edge,
    method=method,
    width=width,
)

pinch_geometric = solution.flux_surface.pinch_velocity(
    rho,
    mixed["diffusion"],
    model="geometric",
    coefficient=0.3,
    rho_edge=rho_edge,
    method=method,
    width=width,
)

pinch_threshold_scan = {}
for nu_th in [0.04, 0.1, 0.3, 1.0, 3.0]:
    pinch_threshold_scan[nu_th] = solution.flux_surface.pinch_velocity(
        rho,
        mixed["diffusion"],
        model="militello",
        coefficient=0.5,
        threshold=nu_th,
        rho_edge=rho_edge,
        method=method,
        width=width,
    )


In [ ]:
collisionality = solution.flux_surface.collisionality(rho, method=method, width=width)
pinch_factor = solution.flux_surface.pinch_factor(rho, method=method, width=width)

q_fs = solution.flux_surface.q(rho, method=method, width=width)
R_fs = solution.flux_surface.major_radius(rho, method=method, width=width)
epsilon_fs = solution.flux_surface.epsilon(rho, method=method, width=width)
ne_fs = solution.flux_surface.average("n", rho, method=method, width=width)
te_fs = solution.flux_surface.te(rho, method=method, width=width)
with np.errstate(divide="ignore", invalid="ignore"):
    ln_lambda_fs = 31.3 - np.log(np.sqrt(ne_fs) / te_fs)
    nu_numerator = q_fs * R_fs * ne_fs * ln_lambda_fs
    nu_denominator = te_fs**2 * epsilon_fs**1.5

summary = {
    "case": case_name,
    "delta_te": mixed["delta_te"],
    "rho_max": rho_full_max,
    "chi_bohm_range": (float(np.nanmin(bohm["chi_bohm"])), float(np.nanmax(bohm["chi_bohm"]))),
    "chi_gyrobohm_range": (float(np.nanmin(gyrobohm["chi_gyrobohm"])), float(np.nanmax(gyrobohm["chi_gyrobohm"]))),
    "chi_i_range": (float(np.nanmin(mixed["chi_i"])), float(np.nanmax(mixed["chi_i"]))),
    "chi_e_range": (float(np.nanmin(mixed["chi_e"])), float(np.nanmax(mixed["chi_e"]))),
    "diffusion_range": (float(np.nanmin(mixed["diffusion"])), float(np.nanmax(mixed["diffusion"]))),
    "collisionality_range": (float(np.nanmin(collisionality)), float(np.nanmax(collisionality))),
    "pinch_factor_range": (float(np.nanmin(pinch_factor)), float(np.nanmax(pinch_factor))),
    "pinch_militello_range": (float(np.nanmin(pinch_militello)), float(np.nanmax(pinch_militello))),
    "pinch_geometric_range": (float(np.nanmin(pinch_geometric)), float(np.nanmax(pinch_geometric))),
}
summary


In [ ]:
sample_rhos = np.array([0.2, 0.5, 0.8, 0.95])
sample_indices = [int(np.nanargmin(np.abs(rho - rho0))) for rho0 in sample_rhos]
collisionality_samples = [
    {
        "rho": float(rho[i]),
        "q": float(q_fs[i]),
        "R_major": float(R_fs[i]),
        "epsilon": float(epsilon_fs[i]),
        "n_e": float(ne_fs[i]),
        "T_e": float(te_fs[i]),
        "lnLambda": float(ln_lambda_fs[i]),
        "numerator": float(nu_numerator[i]),
        "denominator": float(nu_denominator[i]),
        "nu_e_star": float(collisionality[i]),
        "pinch_factor": float(pinch_factor[i]),
    }
    for i in sample_indices
]
collisionality_samples


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)

axes[0, 0].plot(rho, q_fs)
axes[0, 0].set_title("Safety factor q")
axes[0, 0].set_xlabel(r"$\rho_{pol,norm}$")
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(rho, R_fs, label="R")
axes[0, 1].plot(rho, epsilon_fs, label=r"$\epsilon$")
axes[0, 1].set_title("Geometry factors")
axes[0, 1].set_xlabel(r"$\rho_{pol,norm}$")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

axes[0, 2].plot(rho, ne_fs, label=r"$n_e$")
axes[0, 2].plot(rho, te_fs, label=r"$T_e$")
axes[0, 2].set_yscale("log")
axes[0, 2].set_title("Surface-averaged plasma state")
axes[0, 2].set_xlabel(r"$\rho_{pol,norm}$")
axes[0, 2].legend()
axes[0, 2].grid(alpha=0.3)

axes[1, 0].plot(rho, ln_lambda_fs)
axes[1, 0].set_title(r"Coulomb log $\ln \Lambda$")
axes[1, 0].set_xlabel(r"$\rho_{pol,norm}$")
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(rho, nu_numerator)
axes[1, 1].set_yscale("log")
axes[1, 1].set_title(r"Numerator $q R n_e \ln \Lambda$")
axes[1, 1].set_xlabel(r"$\rho_{pol,norm}$")
axes[1, 1].grid(alpha=0.3)

axes[1, 2].plot(rho, nu_denominator, label=r"$T_e^2 \epsilon^{3/2}$")
axes[1, 2].plot(rho, collisionality, label=r"$\nu_e^*$")
axes[1, 2].set_yscale("log")
axes[1, 2].set_title("Collisionality balance")
axes[1, 2].set_xlabel(r"$\rho_{pol,norm}$")
axes[1, 2].legend()
axes[1, 2].grid(alpha=0.3)

plt.show()


## Plot the 1D profiles

This is the first comparison surface to inspect before introducing the local 2D diagnostic transport path.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)

axes[0, 0].plot(rho, bohm["chi_bohm"], label="Bohm")
axes[0, 0].set_yscale("log")
axes[0, 0].set_xlabel(r"$\rho_{pol,norm}$")
axes[0, 0].set_ylabel(r"m$^2$/s")
axes[0, 0].set_title("Bohm component")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(rho, gyrobohm["chi_gyrobohm"], label="Gyro-Bohm")
axes[0, 1].set_yscale("log")
axes[0, 1].set_xlabel(r"$\rho_{pol,norm}$")
axes[0, 1].set_ylabel(r"m$^2$/s")
axes[0, 1].set_title("Gyro-Bohm component")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

axes[0, 2].plot(rho, mixed["chi_i"], label=r"$\chi_i$ ion heat diffusivity")
axes[0, 2].plot(rho, mixed["chi_e"], label=r"$\chi_e$ electron heat diffusivity")
axes[0, 2].plot(rho, mixed["diffusion"], label="D particle diffusion")
axes[0, 2].set_xlabel(r"$\rho_{pol,norm}$")
axes[0, 2].set_ylabel(r"m$^2$/s")
axes[0, 2].set_title("Transport outputs")
axes[0, 2].legend()
axes[0, 2].grid(alpha=0.3)

axes[1, 0].plot(rho, collisionality, label=r"$\nu_e^*$")
axes[1, 0].set_yscale("log")
axes[1, 0].set_xlabel(r"$\rho_{pol,norm}$")
axes[1, 0].set_title("Collisionality")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(rho, np.maximum(pinch_factor, 1e-300), label="Militello factor")
axes[1, 1].set_yscale("log")
axes[1, 1].set_xlabel(r"$\rho_{pol,norm}$")
axes[1, 1].set_title("Pinch factor")
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

axes[1, 2].plot(rho, np.maximum(np.abs(pinch_militello), 1e-300), label="Militello |v_pinch|")
axes[1, 2].plot(rho, np.maximum(np.abs(pinch_geometric), 1e-300), label="Geometric |v_pinch|")
axes[1, 2].set_yscale("log")
axes[1, 2].set_xlabel(r"$\rho_{pol,norm}$")
axes[1, 2].set_title("Pinch comparison")
axes[1, 2].legend()
axes[1, 2].grid(alpha=0.3)

fig.suptitle(f"Transport profiles: {case_name}")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

for nu_th, profile in pinch_threshold_scan.items():
    axes[0].plot(rho, np.maximum(np.abs(profile), 1e-300), label=fr"$\nu_{{th}}={nu_th}$")
axes[0].set_yscale("log")
axes[0].set_xlabel(r"$\rho_{pol,norm}$")
axes[0].set_ylabel(r"m/s")
axes[0].set_title("Militello pinch threshold scan")
axes[0].legend()
axes[0].grid(alpha=0.3)

for nu_th in pinch_threshold_scan:
    factor_scan = solution.flux_surface.pinch_factor(rho, threshold=nu_th, method=method, width=width)
    axes[1].plot(rho, np.maximum(factor_scan, 1e-300), label=fr"$\nu_{{th}}={nu_th}$")
axes[1].set_yscale("log")
axes[1].set_xlabel(r"$\rho_{pol,norm}$")
axes[1].set_title("Pinch-factor threshold scan")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.show()


## Project the 1D surface-derived profiles back to 2D

This keeps the transport model surface-based, but already lets you inspect the corresponding 2D fields on the mesh.


In [ ]:
chi_i_node = solution.flux_surface.project(rho, mixed["chi_i"], target="node")
chi_e_node = solution.flux_surface.project(rho, mixed["chi_e"], target="node")
d_node = solution.flux_surface.project(rho, mixed["diffusion"], target="node")
pinch_militello_node = solution.flux_surface.project(rho, pinch_militello, target="node")
pinch_geometric_node = solution.flux_surface.project(rho, pinch_geometric, target="node")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 10), constrained_layout=True)

solution.mesh.plot.full(
    chi_i_node,
    ax=axes[0, 0],
    connectivity=solution.mesh.geometry.connectivity_big,
    n_levels=80,
    label=r"$\chi_i$ [m$^2$/s]",
)
axes[0, 0].set_title("Projected ion diffusivity")

solution.mesh.plot.full(
    chi_e_node,
    ax=axes[0, 1],
    connectivity=solution.mesh.geometry.connectivity_big,
    n_levels=80,
    label=r"$\chi_e$ [m$^2$/s]",
)
axes[0, 1].set_title("Projected electron diffusivity")

solution.mesh.plot.full(
    d_node,
    ax=axes[0, 2],
    connectivity=solution.mesh.geometry.connectivity_big,
    n_levels=80,
    label=r"$D$ [m$^2$/s]",
)
axes[0, 2].set_title("Projected particle diffusion")

solution.mesh.plot.full(
    pinch_militello_node,
    ax=axes[1, 0],
    connectivity=solution.mesh.geometry.connectivity_big,
    n_levels=80,
    label=r"$v_{pinch,militello}$ [m/s]",
)
axes[1, 0].set_title("Projected Militello pinch")

solution.mesh.plot.full(
    pinch_geometric_node,
    ax=axes[1, 1],
    connectivity=solution.mesh.geometry.connectivity_big,
    n_levels=80,
    label=r"$v_{pinch,geom}$ [m/s]",
)
axes[1, 1].set_title("Projected geometric pinch")

axes[1, 2].axis("off")

plt.show()


## Inspect the gradient assumptions behind the current transport model

These are the surface-averaged normalized gradients built from HDG physical gradients projected along the local flux-normal direction.


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5), constrained_layout=True)
ax.plot(rho, mixed["a_over_te_dte"], label=r"$-a \, \nabla T_e \cdot \hat{n}_\psi / T_e$")
ax.plot(rho, mixed["a_over_pe_dpe"], label=r"$-a \, \nabla p_e \cdot \hat{n}_\psi / p_e$")
ax.set_xlabel(r"$\rho_{pol,norm}$")
ax.set_title(f"Gradient inputs ({mixed['derivative_mode']})")
ax.legend()
ax.grid(alpha=0.3)
plt.show()


## Next comparison step

The missing comparison branch is the fully local 2D diagnostic transport path, where the same `delta_te` is kept but the other ingredients are evaluated locally before plotting. Once that exists, this notebook can be extended to show:
- surface-based projected 2D fields
- fully local 2D fields
- ratio and difference maps between the two assumptions
